# 概率融合

按 ID 对齐，再对调参 baseline 与 TextCNN 的三类 OOF 概率做等权平均。保留集不参与评分。组合在开发结果已知后提出，区间基于当前 OOF 的分组重抽样。

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").is_file())
sys.path.insert(0, str(ROOT))


In [ ]:
import pandas as pd
from src.project_review import comparison, development_data, paired_group_interval, score

dev = development_data()
comparison_table, params, probabilities = comparison(tuned=True)
display(params)
display(comparison_table.assign(delta_vs_fixed=comparison_table["macro_f1"] - score(dev, probabilities["Baseline"], "fixed")["macro_f1"]))
effects = []
for label in ("TextCNN", "Equal-weight vote"):
    left, right = paired_group_interval(dev, probabilities[label], probabilities["Tuned baseline"])
    delta = score(dev, probabilities[label], label)["macro_f1"] - score(dev, probabilities["Tuned baseline"], "tuned")["macro_f1"]
    effects.append({"model": label, "delta_vs_tuned": delta, "95% lower": left, "95% upper": right})
display(pd.DataFrame(effects))
